In [1]:
!hdfs dfs -ls /dataset/calidad_aire_madrid
!hdfs dfs -ls /dataset/magnitudes

Found 2 items
-rw-r--r--   3 root supergroup   20166384 2026-10-01 12:24 /dataset/calidad_aire_madrid/calidad_aire_madrid.csv
-rw-r--r--   3 root supergroup    3828371 2026-10-01 13:19 /dataset/calidad_aire_madrid/calidad_aire_madrid_20000.csv
Found 1 items
-rw-r--r--   3 root supergroup       1142 2026-10-09 10:01 /dataset/magnitudes/magnitudes.csv


In [2]:
    
#Crear la tabla desde Jupyter mediante Beeline

!hdfs dfs -mkdir -p /user/hive/warehouse
!hdfs dfs -chmod 777 /user/hive/warehouse

!beeline -u jdbc:hive2://localhost:10000 -e "CREATE DATABASE IF NOT EXISTS calidad_aire;"


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261009100202_b07cf081-b02a-472d-ba79-eea5e7c7cf8a): CREATE DATABASE IF NOT EXISTS calidad_aire
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:null, properties:null)
INFO  : Completed co

In [3]:
#Crear tabla datos_calidad_aire y cargar datos procendentes del .csv  /dataset/calidad_aire_madrid/calidad_aire_madrid.csv
import subprocess

sql = """
CREATE TABLE calidad_aire.datos_calidad_aire (
    provincia STRING,
    municipio STRING,
    estacion STRING,
    magnitud INT,
    punto_muestreo STRING,
    ano INT,
    mes INT,
    dia INT,
    h01 DOUBLE,
    v01 STRING,
    h02 DOUBLE,
    v02 STRING,
    h03 DOUBLE,
    v03 STRING,
    h04 DOUBLE,
    v04 STRING,
    h05 DOUBLE,
    v05 STRING,
    h06 DOUBLE,
    v06 STRING,
    h07 DOUBLE,
    v07 STRING,
    h08 DOUBLE,
    v08 STRING,
    h09 DOUBLE,
    v09 STRING,
    h10 DOUBLE,
    v10 STRING,
    h11 DOUBLE,
    v11 STRING,
    h12 DOUBLE,
    v12 STRING,
    h13 DOUBLE,
    v13 STRING,
    h14 DOUBLE,
    v14 STRING,
    h15 DOUBLE,
    v15 STRING,
    h16 DOUBLE,
    v16 STRING,
    h17 DOUBLE,
    v17 STRING,
    h18 DOUBLE,
    v18 STRING,
    h19 DOUBLE,
    v19 STRING,
    h20 DOUBLE,
    v20 STRING,
    h21 DOUBLE,
    v21 STRING,
    h22 DOUBLE,
    v22 STRING,
    h23 DOUBLE,
    v23 STRING,
    h24 DOUBLE,
    v24 STRING
)
ROW FORMAT SERDE "org.apache.hadoop.hive.serde2.OpenCSVSerde"
WITH SERDEPROPERTIES (
    "separatorChar" = ";",
    "quoteChar" = '"'
)
STORED AS TEXTFILE
LOCATION 'hdfs://namenode:9000/dataset/calidad_aire_madrid'
"""

sqlbis = """
CREATE TABLE calidad_aire.magnitudes (
    MAGNITUD STRING,
    NOMBRE STRING,
    ABREVIATURA STRING,
    UNIDAD STRING
)
ROW FORMAT SERDE "org.apache.hadoop.hive.serde2.OpenCSVSerde"
WITH SERDEPROPERTIES (
    "separatorChar" = ";",
    "quoteChar" = '"'
)
STORED AS TEXTFILE
LOCATION 'hdfs://namenode:9000/dataset/magnitudes'
"""
resultado_aire = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

resultado_magnitudes = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sqlbis],
    capture_output=True,
    text=True
)


print(resultado_aire.stdout)
print(resultado_aire.stderr)

print(resultado_magnitudes.stdout)
print(resultado_magnitudes.stderr)


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261009100215_049ab32a-222f-42df-a1cf-25fcabcde7a8): CREATE TABLE calidad_aire.datos_calidad_aire (
    provincia STRING,
    municipio STRING,
    estacion STRING,
    magnitud INT,
    punto_muestreo STRING,
    ano INT,
    mes INT,
    dia INT,
    h01 DOUBLE,
    v01 STRING,
    h02 DOUBLE,
    v02 STRI

In [9]:
#Crear tabla datos_calidad_aire 
import time

sql = '''
CREATE TABLE calidad_aire.datos_calidad_aire_parquet
STORED AS PARQUET
AS
SELECT *
FROM calidad_aire.datos_calidad_aire
;
'''

inicio = time.time()

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261008113210_62659b71-3b33-4d9c-a097-24ccf80fd220): CREATE TABLE calidad_aire.datos_calidad_aire_parquet
STORED AS PARQUET
AS
SELECT *
FROM calidad_aire.datos_calidad_aire
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returnin

In [29]:
#Crear tabla datos_calidad_aire 
import time

sql = '''
CREATE TABLE calidad_aire.magnitudes_parquet
STORED AS PARQUET
AS
SELECT *
FROM calidad_aire.magnitudes
;
'''

inicio = time.time()

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261008122041_56dfbf1d-ad2b-42d8-be94-e6aff907de59): CREATE TABLE calidad_aire.magnitudes_parquet
STORED AS PARQUET
AS
SELECT *
FROM calidad_aire.magnitudes
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: S

In [4]:
#Consultar la tabla creada
!beeline -u jdbc:hive2://localhost:10000 -e "USE calidad_aire; SHOW TABLES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261009100259_47cb757a-b8f0-43be-9b6e-2385c037ab2c): USE calidad_aire
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:null, properties:null)
INFO  : Completed compiling command(queryId=ro

In [14]:
# Cargar el CSV en la tabla
import subprocess

sql = """
USE calidad_aire;

SELECT COUNT(*) AS total_registros_aire
FROM datos_calidad_aire;

SELECT COUNT(*) AS total_registros_magnitudes
FROM magnitudes;
"""

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

# Mostrar únicamente las últimas líneas relevantes
print(resultado.stdout[-2000:])

+-----------------------+
| total_registros_aire  |
+-----------------------+
| 96515                 |
+-----------------------+
+-----------------------------+
| total_registros_magnitudes  |
+-----------------------------+
| 20                          |
+-----------------------------+



In [1]:
#Si detectamos algo erroneo, podemos eliminar la tabla deseada y volver al paso de creación de tabla y de carga de datos
import subprocess

sql = """
DROP TABLE IF EXISTS calidad_aire.magnitudes
"""

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)



SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/local/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/local/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Connecting to jdbc:hive2://localhost:10000
Connected to: Apache Hive (version 3.1.3)
Driver: Hive JDBC (version 3.1.3)
Transaction isolation: TRANSACTION_REPEATABLE_READ
INFO  : Compiling command(queryId=root_20261006072153_62234184-3822-47c5-a9f8-23071b1afa11): DROP TABLE IF EXISTS calidad_aire.magnitudes
INFO  : Concurrency mode is disabled, not creating a lock manager
INFO  : Semantic Analysis Completed (retrial = false)
INFO  : Returning Hive schema: Schema(fieldSchemas:null, properties:null)
INFO  : Completed

In [10]:
#Si detectamos algo erroneo, podemos eliminar la tabla deseada y volver al paso de creación de tabla y de carga de datos
import subprocess

sql = """
SELECT COUNT(*)
FROM calidad_aire.datos_calidad_aire_parquet;
"""

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)


+--------+
|  _c0   |
+--------+
| 96515  |
+--------+



In [40]:

# Comparar los tiempos de ejecución entre formato csv y parquet
import time
sql = """
SELECT
    CAST(TRIM(d.MAGNITUD) AS INT) AS cod_magnitud,
    m.nombre AS contaminante,
    m.abreviatura AS abreviatura,
    m.unidad AS unidad,
    COUNT(*) AS registros
FROM calidad_aire.datos_calidad_aire d
LEFT JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.magnitud) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
  AND TRIM(m.magnitud) <> 'Cod. Magnitud'
GROUP BY
    CAST(TRIM(d.MAGNITUD) AS INT),
    m.nombre,
    m.abreviatura,
    m.unidad
ORDER BY
    CAST(TRIM(d.MAGNITUD) AS INT);
"""

inicio = time.time()

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")



+---------------+------------------------+--------------+---------+------------+
| cod_magnitud  |      contaminante      | abreviatura  | unidad  | registros  |
+---------------+------------------------+--------------+---------+------------+
| 1             | Di?xido de Azufre      | SO2          | ?g/m3   | 2432       |
| 6             | Mon?xido de Carbono    | CO           | mg/m3   | 2431       |
| 7             | Mon?xido de Nitr?geno  | NO           | ?g/m3   | 17627      |
| 8             | Di?xido de Nitr?geno   | NO2          | ?g/m3   | 17627      |
| 9             | Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 7288       |
| 10            | Part?culas < 10 ?m     | PM10         | ?g/m3   | 9715       |
| 12            | ?xidos de Nitr?geno    | NOx          | ?g/m3   | 17638      |
| 14            | Ozono                  | O3           | ?g/m3   | 10940      |
| 20            | Tolueno                | TOL          | ?g/m3   | 3605       |
| 30            | Benceno   

In [41]:

# Comparar los tiempos de ejecución entre formato csv y parquet
import time
sql = """
SELECT
    CAST(TRIM(d.MAGNITUD) AS INT) AS cod_magnitud,
    m.nombre AS contaminante,
    m.abreviatura AS abreviatura,
    m.unidad AS unidad,
    COUNT(*) AS registros
FROM calidad_aire.datos_calidad_aire_parquet d
LEFT JOIN calidad_aire.magnitudes_parquet m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.magnitud) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
  AND TRIM(m.magnitud) <> 'Cod. Magnitud'
GROUP BY
    CAST(TRIM(d.MAGNITUD) AS INT),
    m.nombre,
    m.abreviatura,
    m.unidad
ORDER BY
    CAST(TRIM(d.MAGNITUD) AS INT);
"""

inicio = time.time()

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")



+---------------+------------------------+--------------+---------+------------+
| cod_magnitud  |      contaminante      | abreviatura  | unidad  | registros  |
+---------------+------------------------+--------------+---------+------------+
| 1             | Di?xido de Azufre      | SO2          | ?g/m3   | 2432       |
| 6             | Mon?xido de Carbono    | CO           | mg/m3   | 2431       |
| 7             | Mon?xido de Nitr?geno  | NO           | ?g/m3   | 17627      |
| 8             | Di?xido de Nitr?geno   | NO2          | ?g/m3   | 17627      |
| 9             | Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 7288       |
| 10            | Part?culas < 10 ?m     | PM10         | ?g/m3   | 9715       |
| 12            | ?xidos de Nitr?geno    | NOx          | ?g/m3   | 17638      |
| 14            | Ozono                  | O3           | ?g/m3   | 10940      |
| 20            | Tolueno                | TOL          | ?g/m3   | 3605       |
| 30            | Benceno   

In [45]:
#Consultas analíticas 1: ¿Que contaminantes aparecen en el conjunto de datos?
import subprocess
import time

sql = """
SELECT
    d.MAGNITUD AS cod_magnitud,
    m.NOMBRE AS nombre,
    m.ABREVIATURA AS abreviatura,
    m.UNIDAD AS unidad,
    COUNT(*) AS registros
FROM calidad_aire.datos_calidad_aire d
LEFT JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.MAGNITUD) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
  AND TRIM(m.MAGNITUD) <> 'Cod. Magnitud'
GROUP BY
    d.MAGNITUD,
    m.NOMBRE,
    m.ABREVIATURA,
    m.UNIDAD
ORDER BY
    CAST(d.MAGNITUD AS INT);

"""
inicio = time.time()

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")




+---------------+------------------------+--------------+---------+------------+
| cod_magnitud  |         nombre         | abreviatura  | unidad  | registros  |
+---------------+------------------------+--------------+---------+------------+
| 1             | Di?xido de Azufre      | SO2          | ?g/m3   | 2432       |
| 6             | Mon?xido de Carbono    | CO           | mg/m3   | 2431       |
| 7             | Mon?xido de Nitr?geno  | NO           | ?g/m3   | 17627      |
| 8             | Di?xido de Nitr?geno   | NO2          | ?g/m3   | 17627      |
| 9             | Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 7288       |
| 10            | Part?culas < 10 ?m     | PM10         | ?g/m3   | 9715       |
| 12            | ?xidos de Nitr?geno    | NOx          | ?g/m3   | 17638      |
| 14            | Ozono                  | O3           | ?g/m3   | 10940      |
| 20            | Tolueno                | TOL          | ?g/m3   | 3605       |
| 30            | Benceno   

In [37]:
#Consultas analíticas 2: Número de registros por magnitud
import subprocess

sql = """
SELECT
    m.NOMBRE AS magnitud,
    m.ABREVIATURA AS abreviatura,
    m.UNIDAD AS unidad,
    COUNT(*) AS registros
FROM calidad_aire.datos_calidad_aire d
JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.MAGNITUD) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
GROUP BY
    m.NOMBRE,
    m.ABREVIATURA,
    m.UNIDAD
ORDER BY registros DESC;

"""
inicio = time.time()
resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

fin = time.time()

print(resultado.stdout)
print(resultado.stderr)

print(f"Tiempo total: {fin - inicio:.2f} segundos")

+------------------------+--------------+---------+------------+
|        magnitud        | abreviatura  | unidad  | registros  |
+------------------------+--------------+---------+------------+
| ?xidos de Nitr?geno    | NOx          | ?g/m3   | 17638      |
| Mon?xido de Nitr?geno  | NO           | ?g/m3   | 17627      |
| Di?xido de Nitr?geno   | NO2          | ?g/m3   | 17627      |
| Ozono                  | O3           | ?g/m3   | 10940      |
| Part?culas < 10 ?m     | PM10         | ?g/m3   | 9715       |
| Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 7288       |
| Tolueno                | TOL          | ?g/m3   | 3605       |
| Etilbenceno            | EBE          | ?g/m3   | 3605       |
| Benceno                | BEN          | ?g/m3   | 3605       |
| Di?xido de Azufre      | SO2          | ?g/m3   | 2432       |
| Mon?xido de Carbono    | CO           | mg/m3   | 2431       |
+------------------------+--------------+---------+------------+

SLF4J: Class path contai

In [5]:
#Consultas analíticas 3: Registros por municipio
# Consulta analítica: número de registros de cada contaminante por municipio

import subprocess

sql = """
SELECT
    d.MUNICIPIO AS municipio,
    m.NOMBRE AS contaminante,
    m.ABREVIATURA AS abreviatura,
    m.UNIDAD AS unidad,
    COUNT(*) AS registros
FROM calidad_aire.datos_calidad_aire d
LEFT JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.MAGNITUD) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
GROUP BY
    d.MUNICIPIO,
    m.NOMBRE,
    m.ABREVIATURA,
    m.UNIDAD
ORDER BY
    d.MUNICIPIO,
    registros DESC;
"""

resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)

+------------+------------------------+--------------+---------+------------+
| municipio  |      contaminante      | abreviatura  | unidad  | registros  |
+------------+------------------------+--------------+---------+------------+
| 079        | ?xidos de Nitr?geno    | NOx          | ?g/m3   | 13981      |
| 079        | Di?xido de Nitr?geno   | NO2          | ?g/m3   | 13981      |
| 079        | Mon?xido de Nitr?geno  | NO           | ?g/m3   | 13981      |
| 079        | Part?culas < 10 ?m     | PM10         | ?g/m3   | 8501       |
| 079        | Ozono                  | O3           | ?g/m3   | 7902       |
| 079        | Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 6682       |
| 079        | Mon?xido de Carbono    | CO           | mg/m3   | 2431       |
| 079        | Benceno                | BEN          | ?g/m3   | 2410       |
| 079        | Tolueno                | TOL          | ?g/m3   | 2410       |
| 079        | Etilbenceno            | EBE          | ?g/m3   |

In [26]:
#Consultas analíticas 4: qué contaminante presenta los valores medios más elevados
import subprocess

sql = """
SELECT
    m.NOMBRE AS contaminante,
    m.ABREVIATURA AS abreviatura,
    m.UNIDAD AS unidad,
    AVG(
        (COALESCE(d.H01,0) + COALESCE(d.H02,0) +
         COALESCE(d.H03,0) + COALESCE(d.H04,0) +
         COALESCE(d.H05,0) + COALESCE(d.H06,0) +
         COALESCE(d.H07,0) + COALESCE(d.H08,0) +
         COALESCE(d.H09,0) + COALESCE(d.H10,0) +
         COALESCE(d.H11,0) + COALESCE(d.H12,0) +
         COALESCE(d.H13,0) + COALESCE(d.H14,0) +
         COALESCE(d.H15,0) + COALESCE(d.H16,0) +
         COALESCE(d.H17,0) + COALESCE(d.H18,0) +
         COALESCE(d.H19,0) + COALESCE(d.H20,0) +
         COALESCE(d.H21,0) + COALESCE(d.H22,0) +
         COALESCE(d.H23,0) + COALESCE(d.H24,0)
        ) / 24.0
    ) AS valor_medio
FROM calidad_aire.datos_calidad_aire d
JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.MAGNITUD) AS INT)
WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'
GROUP BY
    m.NOMBRE,
    m.ABREVIATURA,
    m.UNIDAD
ORDER BY valor_medio DESC;
"""
inicio = time.time()
resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)

+------------------------+--------------+---------+---------------------+
|      contaminante      | abreviatura  | unidad  |     valor_medio     |
+------------------------+--------------+---------+---------------------+
| Ozono                  | O3           | ?g/m3   | 61.91869667885422   |
| ?xidos de Nitr?geno    | NOx          | ?g/m3   | 28.84583238840373   |
| Di?xido de Nitr?geno   | NO2          | ?g/m3   | 20.69360214443749   |
| Part?culas < 10 ?m     | PM10         | ?g/m3   | 18.808427689140498  |
| Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 8.84172040797657    |
| Mon?xido de Nitr?geno  | NO           | ?g/m3   | 5.2968717497778455  |
| Di?xido de Azufre      | SO2          | ?g/m3   | 3.470086348684206   |
| Tolueno                | TOL          | ?g/m3   | 1.932889505316689   |
| Etilbenceno            | EBE          | ?g/m3   | 0.3501236708275542  |
| Benceno                | BEN          | ?g/m3   | 0.3405802126675915  |
| Mon?xido de Carbono    | CO         

In [6]:
#Consultas analíticas 5: Contaminante que más veces supera un umbral de referencia
import subprocess
import time

sql = """
SELECT
    m.NOMBRE AS contaminante,
    m.ABREVIATURA AS abreviatura,
    m.UNIDAD AS unidad,

    SUM(
        CASE
            WHEN d.MAGNITUD = '10' AND d.H01 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H02 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H03 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H04 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H05 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H06 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H07 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H08 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H09 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H10 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H11 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H12 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H13 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H14 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H15 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H16 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H17 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H18 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H19 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H20 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H21 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H22 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H23 > 50 THEN 1
            WHEN d.MAGNITUD = '10' AND d.H24 > 50 THEN 1

            WHEN d.MAGNITUD = '9' AND d.H01 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H02 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H03 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H04 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H05 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H06 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H07 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H08 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H09 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H10 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H11 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H12 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H13 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H14 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H15 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H16 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H17 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H18 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H19 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H20 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H21 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H22 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H23 > 25 THEN 1
            WHEN d.MAGNITUD = '9' AND d.H24 > 25 THEN 1

            WHEN d.MAGNITUD = '8' AND d.H01 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H02 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H03 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H04 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H05 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H06 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H07 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H08 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H09 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H10 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H11 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H12 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H13 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H14 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H15 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H16 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H17 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H18 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H19 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H20 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H21 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H22 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H23 > 200 THEN 1
            WHEN d.MAGNITUD = '8' AND d.H24 > 200 THEN 1
            ELSE 0
        END
    ) AS superaciones

FROM calidad_aire.datos_calidad_aire d
JOIN calidad_aire.magnitudes m
    ON CAST(TRIM(d.MAGNITUD) AS INT) =
       CAST(TRIM(m.MAGNITUD) AS INT)

WHERE TRIM(d.MAGNITUD) <> 'MAGNITUD'

GROUP BY
    m.NOMBRE,
    m.ABREVIATURA,
    m.UNIDAD

ORDER BY superaciones DESC;
"""
inicio = time.time()
resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)

+------------------------+--------------+---------+---------------+
|      contaminante      | abreviatura  | unidad  | superaciones  |
+------------------------+--------------+---------+---------------+
| Part?culas < 10 ?m     | PM10         | ?g/m3   | 2147          |
| Part?culas < 2.5 ?m    | PM2.5        | ?g/m3   | 1269          |
| ?xidos de Nitr?geno    | NOx          | ?g/m3   | 0             |
| Tolueno                | TOL          | ?g/m3   | 0             |
| Ozono                  | O3           | ?g/m3   | 0             |
| Mon?xido de Nitr?geno  | NO           | ?g/m3   | 0             |
| Mon?xido de Carbono    | CO           | mg/m3   | 0             |
| Etilbenceno            | EBE          | ?g/m3   | 0             |
| Di?xido de Nitr?geno   | NO2          | ?g/m3   | 0             |
| Di?xido de Azufre      | SO2          | ?g/m3   | 0             |
| Benceno                | BEN          | ?g/m3   | 0             |
+------------------------+--------------+-------

In [32]:
#Consultas analíticas 6: ¿Qué contaminante supera con mayor frecuencia su valor de referencia?
import subprocess

sql = """
SELECT
    CASE
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 10 THEN 'PM10'
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 9 THEN 'PM2.5'
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 8 THEN 'NO2'
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 14 THEN 'O3'
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 1 THEN 'SO2'
    END AS contaminante,

    CASE
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 10 THEN 50
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 9 THEN 25
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 8 THEN 200
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 14 THEN 180
        WHEN CAST(TRIM(MAGNITUD) AS INT) = 1 THEN 350
    END AS valor_referencia,

    SUM(
        CASE WHEN CAST(H01 AS DOUBLE) >
            CASE
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 10 THEN 50
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 9 THEN 25
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 8 THEN 200
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 14 THEN 180
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 1 THEN 350
            END
        THEN 1 ELSE 0 END
    )
    +
    SUM(
        CASE WHEN CAST(H02 AS DOUBLE) >
            CASE
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 10 THEN 50
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 9 THEN 25
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 8 THEN 200
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 14 THEN 180
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 1 THEN 350
            END
        THEN 1 ELSE 0 END
    )
    +
    SUM(
        CASE WHEN CAST(H03 AS DOUBLE) >
            CASE
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 10 THEN 50
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 9 THEN 25
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 8 THEN 200
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 14 THEN 180
                WHEN CAST(TRIM(MAGNITUD) AS INT) = 1 THEN 350
            END
        THEN 1 ELSE 0 END
    )

FROM calidad_aire.datos_calidad_aire

WHERE CAST(TRIM(MAGNITUD) AS INT) IN (1,8,9,10,14)

GROUP BY MAGNITUD
ORDER BY contaminante;
"""
inicio = time.time()
resultado = subprocess.run(
    ["beeline", "-u", "jdbc:hive2://localhost:10000", "-e", sql],
    capture_output=True,
    text=True
)

print(resultado.stdout)

+---------------+-------------------+-------+
| contaminante  | valor_referencia  |  _c2  |
+---------------+-------------------+-------+
| NO2           | 200               | 0     |
| O3            | 180               | 0     |
| PM10          | 50                | 1114  |
| PM2.5         | 25                | 808   |
| SO2           | 350               | 0     |
+---------------+-------------------+-------+

